# install deps
- opencv-python: to read and process images and videos
- matplotlib: to visualize images and videos, render the results so that we can see outputs of our preprocessed videos
- imageio: to create a gif from the processed video frames to see a couple of frames stacked together
- gdown: to download the dataset from google drive
- tensorflow: to build and train the DNN

In [ ]:
%pip install opencv-python matplotlib imageio gdown tensorflow

In [ ]:
!pip list

In [ ]:
import os
import cv2 # to read and process videos
import tensorflow as tf # primary DL framework, for data pipeline and model building
import numpy as np # to preprocess any arrays
from typing import List # type annotation
from matplotlib import pyplot as plt # to render our post-process output of our data loading function
import imageio # convert numpy array to gif

# 1. build data loading function

We are going to use GRID dataset for our lip reading model. This dataset is something that can later be used with a custom dataset, say videos of ourselves. We just need to capture frames of a person speaking, then use a speech to text model to transcribe what the person is saying. That dataset can then be subbed into the model training pipeline. We need two data loading functions, one to load the videos and another to load the aligned transcriptions.

In [14]:
import gdown

In [ ]:
url = 'https://drive.google.com/uc?id=1YlvpDLix3S-U8fd-gqRwPcWXAXm8JwjL'
output = 'data.zip'
gdown.download(url, output, quiet=False)
gdown.extractall('data.zip')

Now we have something here called alignments in the dataset and there is .align file for each video, which is like annotations for their corresponding video. The .align file contains the start and end times of each word spoken in the video, along with the actual word. This is crucial for training our lip reading model, as it allows us to know exactly which frames correspond to which words.

Here sil is silence, bin is the word "bin", blue is the word "blue", at is the word "at", f is the word "f", two is the word "two", and now is the word "now". The numbers represent the start and end times of each word in milliseconds.

We got to train a model that is able to decode this purely from a vid and no audio. We'll remove the sil part and we want to purely extract the middle part 
```
0 23750 sil
23750 29500 bin
29500 34000 blue
34000 35500 at
35500 41000 f
41000 47250 two
47250 53000 now
53000 74500 sil
```